# Module 5 • Neural Networks for Natural Language Processing

# Lesson 31 • Self-Attention and Transformer Architecture Foundations

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 160–200 minutes  
**Execution target:** CPU

---

## Scope

This lesson introduces the core Transformer architecture. It develops
self-attention, scaled dot-product attention, attention masks, multi-head
attention, positional encoding, residual connections, layer normalization,
position-wise feed-forward networks, Transformer encoder blocks, and a compact
CPU-only text classifier implemented with PyTorch.

No external dataset or pretrained model download is required.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain why self-attention differs from recurrent processing;
- define queries, keys, and values in self-attention;
- calculate scaled dot-product attention;
- explain the role of the scaling factor;
- apply padding and causal masks;
- explain multi-head attention;
- track tensor shapes across heads;
- implement sinusoidal positional encoding;
- explain residual connections and layer normalization;
- describe position-wise feed-forward networks;
- assemble a Transformer encoder block;
- build a Transformer text classifier;
- inspect attention-compatible representations;
- compare Transformer and recurrent computational properties;
- discuss Arabic and multilingual tokenization considerations.

## Table of Contents

1. From Recurrent Attention to Self-Attention
2. Why Transformers?
3. Query, Key, and Value Projections
4. Scaled Dot-Product Attention
5. Why Scale by the Key Dimension?
6. Padding Masks
7. Causal Masks
8. Manual NumPy Self-Attention
9. Multi-Head Attention
10. Head Concatenation
11. Positional Information
12. Sinusoidal Positional Encoding
13. Learned Positional Embeddings
14. Residual Connections
15. Layer Normalization
16. Position-Wise Feed-Forward Networks
17. Transformer Encoder Block
18. Pre-Norm Versus Post-Norm
19. Computational Complexity
20. Dataset
21. Splits and Vocabulary
22. Dataset and Dynamic Padding
23. Positional Encoding Module
24. Transformer Encoder Classifier
25. Shape Inspection
26. Mask Inspection
27. Training Utilities
28. Training the Classifier
29. Learning Curves
30. Evaluation
31. Confusion Matrix
32. Error Analysis
33. Representation Inspection
34. Self-Attention Versus RNNs
35. Common Failure Modes
36. Arabic and Multilingual Considerations
37. Reproducibility and Reporting
38. Knowledge Check
39. Exercises
40. Summary and Next Lesson

# 1. From Recurrent Attention to Self-Attention

In recurrent encoder–decoder attention, the decoder queries encoder states.

In self-attention, tokens within the same sequence query one another.

```text
token sequence
     ↓
query/key/value projections
     ↓
token-to-token attention
     ↓
contextual token representations
```

In [ ]:
import copy
import random
import re
from collections import Counter
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

architecture_summary = pd.DataFrame(
    [
        ("RNN", "sequential recurrence", "limited"),
        ("CNN", "local convolution", "high"),
        ("Transformer", "global self-attention", "high"),
    ],
    columns=["Architecture", "Context mechanism", "Parallelism"],
)

architecture_summary

# 2. Why Transformers?

Transformers avoid recurrence across sequence positions.

Benefits:

- strong parallelism during training;
- direct paths between distant tokens;
- flexible global context;
- scalable encoder and decoder stacks.

Costs:

- quadratic attention cost in sequence length;
- explicit positional information is required;
- large data and compute requirements at scale.

# 3. Query, Key, and Value Projections

Given token representations \(X\):

\[
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
\]

Each token becomes:

- a query;
- a key;
- a value.

In [ ]:
rng = np.random.default_rng(42)

sequence_length = 4
model_dimension = 6
key_dimension = 3
value_dimension = 3

X = rng.normal(
    size=(sequence_length, model_dimension)
)

W_Q = rng.normal(
    0.0,
    0.2,
    size=(model_dimension, key_dimension),
)
W_K = rng.normal(
    0.0,
    0.2,
    size=(model_dimension, key_dimension),
)
W_V = rng.normal(
    0.0,
    0.2,
    size=(model_dimension, value_dimension),
)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

# 4. Scaled Dot-Product Attention

\[
Attention(Q,K,V)
=
softmax\left(
\frac{QK^T}{\sqrt{d_k}}
ight)V
\]

In [ ]:
def softmax_numpy(
    values: np.ndarray,
    axis: int = -1,
) -> np.ndarray:
    shifted = (
        values
        - np.max(
            values,
            axis=axis,
            keepdims=True,
        )
    )

    exponentials = np.exp(shifted)

    return (
        exponentials
        / exponentials.sum(
            axis=axis,
            keepdims=True,
        )
    )


scores = (
    Q @ K.T
    / math.sqrt(key_dimension)
)

weights = softmax_numpy(
    scores,
    axis=1,
)

attended = weights @ V

print("Scores:", scores.shape)
print("Weights:", weights.shape)
print("Output:", attended.shape)
print("Weight row sums:", weights.sum(axis=1))

# 5. Why Scale by the Key Dimension?

Dot products grow in magnitude as key dimension increases. Large logits push
softmax toward saturation, producing small gradients.

Scaling by \(\sqrt{d_k}\) controls logit magnitude.

In [ ]:
dimensions = [4, 16, 64, 256]
rows = []

for dimension in dimensions:
    q = rng.normal(size=(500, dimension))
    k = rng.normal(size=(500, dimension))

    raw = np.sum(q * k, axis=1)
    scaled = raw / math.sqrt(dimension)

    rows.append(
        {
            "dimension": dimension,
            "raw_std": raw.std(),
            "scaled_std": scaled.std(),
        }
    )

pd.DataFrame(rows)

# 6. Padding Masks

Padding positions must not contribute attention.

A key padding mask blocks padded keys for every query.

In [ ]:
example_scores = np.array(
    [
        [0.4, 1.0, 0.2, -0.3],
        [0.8, 0.1, 0.5, 0.0],
    ]
)

valid_keys = np.array(
    [
        [True, True, True, False],
        [True, True, False, False],
    ]
)

masked_scores = np.where(
    valid_keys,
    example_scores,
    -1e9,
)

masked_weights = softmax_numpy(
    masked_scores,
    axis=1,
)

masked_weights

# 7. Causal Masks

A causal mask prevents a position from attending to future positions.

It is required for autoregressive Transformer decoders.

In [ ]:
causal_mask = np.tril(
    np.ones(
        (6, 6),
        dtype=int,
    )
)

pd.DataFrame(causal_mask)

Encoder self-attention normally uses padding masks but not causal masks.

# 8. Manual NumPy Self-Attention

In [ ]:
def scaled_dot_product_attention_numpy(
    queries: np.ndarray,
    keys: np.ndarray,
    values: np.ndarray,
    key_mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    logits = (
        queries @ keys.swapaxes(-1, -2)
        / math.sqrt(queries.shape[-1])
    )

    if key_mask is not None:
        logits = np.where(
            key_mask[:, None, :],
            logits,
            -1e9,
        )

    attention_weights = softmax_numpy(
        logits,
        axis=-1,
    )

    output = (
        attention_weights
        @ values
    )

    return output, attention_weights


batch_queries = rng.normal(
    size=(2, 4, 3)
)
batch_keys = rng.normal(
    size=(2, 4, 3)
)
batch_values = rng.normal(
    size=(2, 4, 5)
)

key_mask = np.array(
    [
        [True, True, True, False],
        [True, True, False, False],
    ]
)

attention_output, attention_weights = (
    scaled_dot_product_attention_numpy(
        batch_queries,
        batch_keys,
        batch_values,
        key_mask,
    )
)

print(attention_output.shape)
print(attention_weights.shape)

# 9. Multi-Head Attention

Multi-head attention divides the model dimension across several heads.

Each head learns a different projection and attention pattern.

In [ ]:
multi_head_shapes = pd.DataFrame(
    [
        ("Input", "(B, T, D)"),
        ("Projected Q/K/V", "(B, T, D)"),
        ("Split heads", "(B, H, T, D/H)"),
        ("Head output", "(B, H, T, D/H)"),
        ("Concatenated", "(B, T, D)"),
    ],
    columns=["Tensor", "Shape"],
)

multi_head_shapes

The model dimension must usually be divisible by the number of heads.

# 10. Head Concatenation

Head outputs are concatenated and projected:

\[
MultiHead(Q,K,V)
=
Concat(head_1,\dots,head_h)W_O
\]

Different heads are not guaranteed to learn human-interpretable linguistic roles.

# 11. Positional Information

Self-attention alone is permutation-equivariant. It does not know token order.

Positional information is therefore added to token embeddings.

# 12. Sinusoidal Positional Encoding

Sinusoidal encoding uses fixed sine and cosine patterns at different frequencies.

In [ ]:
def sinusoidal_encoding_numpy(
    maximum_length: int,
    model_dimension: int,
) -> np.ndarray:
    positions = np.arange(
        maximum_length
    )[:, None]

    dimensions = np.arange(
        model_dimension
    )[None, :]

    angle_rates = 1.0 / np.power(
        10000.0,
        (
            2
            * (dimensions // 2)
            / model_dimension
        ),
    )

    angles = positions * angle_rates

    encoding = np.zeros(
        (maximum_length, model_dimension)
    )

    encoding[:, 0::2] = np.sin(
        angles[:, 0::2]
    )
    encoding[:, 1::2] = np.cos(
        angles[:, 1::2]
    )

    return encoding


positional_matrix = sinusoidal_encoding_numpy(
    maximum_length=30,
    model_dimension=16,
)

print(positional_matrix.shape)

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(
    positional_matrix,
    aspect="auto",
)
plt.xlabel("Embedding dimension")
plt.ylabel("Position")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

# 13. Learned Positional Embeddings

Learned position embeddings assign trainable vectors to positions.

Comparison:

- sinusoidal: fixed and extrapolatable in principle;
- learned: task-adaptive but bounded by trained position range.

In [ ]:
positional_comparison = pd.DataFrame(
    [
        ("Sinusoidal", False, "fixed mathematical pattern"),
        ("Learned", True, "position lookup table"),
    ],
    columns=["Type", "Trainable", "Mechanism"],
)

positional_comparison

# 14. Residual Connections

A residual connection adds a sublayer input back to its output:

\[
y = x + Sublayer(x)
\]

Residual paths support optimization and information flow.

# 15. Layer Normalization

Layer normalization normalizes feature dimensions within each token
representation.

It differs from batch normalization because it does not depend on batch-level
statistics.

In [ ]:
layer_norm = nn.LayerNorm(
    normalized_shape=6
)

tensor_example = torch.randn(
    2,
    4,
    6,
)

normalized = layer_norm(
    tensor_example
)

print(normalized.shape)
print(
    "Mean across features:",
    normalized.mean(dim=-1).round(decimals=4),
)

# 16. Position-Wise Feed-Forward Networks

The same feed-forward network is applied independently to every position:

\[
FFN(x) = W_2 activation(W_1x + b_1) + b_2
\]

In [ ]:
feed_forward = nn.Sequential(
    nn.Linear(16, 64),
    nn.ReLU(),
    nn.Linear(64, 16),
)

ffn_input = torch.randn(
    3,
    7,
    16,
)

print(
    feed_forward(ffn_input).shape
)

Self-attention mixes information across positions. The feed-forward network
transforms features within each position.

# 17. Transformer Encoder Block

A Transformer encoder block contains:

1. multi-head self-attention;
2. residual connection and normalization;
3. position-wise feed-forward network;
4. another residual connection and normalization.

In [ ]:
encoder_block_summary = pd.DataFrame(
    [
        (1, "Multi-head self-attention"),
        (2, "Residual connection"),
        (3, "Layer normalization"),
        (4, "Feed-forward network"),
        (5, "Residual connection"),
        (6, "Layer normalization"),
    ],
    columns=["Stage", "Operation"],
)

encoder_block_summary

# 18. Pre-Norm Versus Post-Norm

- **Post-norm:** normalization follows residual addition.
- **Pre-norm:** normalization precedes each sublayer.

Modern implementations often favor pre-norm for deeper training stability.

# 19. Computational Complexity

Full self-attention compares all pairs of positions.

Approximate attention-matrix cost:

\[
O(T^2)
\]

where \(T\) is sequence length.

In [ ]:
lengths = np.array(
    [16, 32, 64, 128, 256]
)

pair_counts = lengths ** 2

pd.DataFrame(
    {
        "sequence_length": lengths,
        "attention_pairs": pair_counts,
    }
)

# 20. Dataset

We use four balanced intent classes:

- health;
- finance;
- technology;
- travel.

In [ ]:
records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medicine to patient", "health"),
    ("patient visits clinic for diagnosis", "health"),
    ("hospital schedules medical treatment", "health"),
    ("exercise supports long term health", "health"),
    ("nutrition improves patient recovery", "health"),
    ("doctor reviews the medical report", "health"),
    ("clinic provides emergency service", "health"),
    ("nurse helps the patient today", "health"),
    ("medicine reduces the health problem", "health"),
    ("hospital needs experienced doctors", "health"),
    ("patient requests treatment information", "health"),
    ("medical team monitors patient recovery", "health"),
    ("doctor confirms the diagnosis later", "health"),
    ("clinic updates the treatment plan", "health"),
    ("patient receives medicine after examination", "health"),

    ("bank approves customer loan", "finance"),
    ("invoice contains payment charge", "finance"),
    ("customer requests card refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased today", "finance"),
    ("bank transfers money safely", "finance"),
    ("payment failed on the card", "finance"),
    ("refund request remains pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("customer updates bank account", "finance"),
    ("billing service changed the charge", "finance"),
    ("loan payment needs approval", "finance"),
    ("bank reviews the financial request", "finance"),
    ("customer receives refund after review", "finance"),
    ("payment system confirms the transaction", "finance"),
    ("card account shows an extra charge", "finance"),

    ("software update caused an error", "technology"),
    ("application cannot reach the server", "technology"),
    ("network upload failed today", "technology"),
    ("computer needs a system update", "technology"),
    ("device cannot install the software", "technology"),
    ("server lost important data", "technology"),
    ("application displays a network error", "technology"),
    ("computer connects to the server", "technology"),
    ("upload request failed again", "technology"),
    ("system update needs technical help", "technology"),
    ("device reports a software problem", "technology"),
    ("network service is unavailable", "technology"),
    ("server restarts after the update", "technology"),
    ("application recovers after installation", "technology"),
    ("computer stores data on server", "technology"),
    ("network error interrupts the upload", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel reservation", "travel"),
    ("airport lost passenger luggage", "travel"),
    ("travel ticket changed today", "travel"),
    ("flight delay affects the journey", "travel"),
    ("hotel reservation needs an update", "travel"),
    ("tourist visits the city museum", "travel"),
    ("beach trip starts tomorrow", "travel"),
    ("airport changes the flight gate", "travel"),
    ("passenger requests travel information", "travel"),
    ("journey includes a hotel stay", "travel"),
    ("ticket service reports a delay", "travel"),
    ("tourist reaches airport before departure", "travel"),
    ("passenger collects luggage after arrival", "travel"),
    ("hotel confirms the reservation", "travel"),
    ("flight continues after a short delay", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 21. Splits and Vocabulary

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


X_train_full, X_test, y_train_full, y_test = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.25,
    random_state=42,
    stratify=dataset["label"],
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

counts = Counter(
    token
    for text in X_train
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
] + sorted(counts)

word_to_index = {
    token: index
    for index, token in enumerate(vocabulary)
}

PAD_ID = word_to_index["<PAD>"]
UNK_ID = word_to_index["<UNK>"]
CLS_ID = word_to_index["<CLS>"]

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

print("Vocabulary size:", len(vocabulary))
print("Train:", len(X_train))
print("Validation:", len(X_validation))
print("Test:", len(X_test))

# 22. Dataset and Dynamic Padding

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(
        self,
        texts,
        labels,
    ):
        self.texts = list(texts)
        self.labels = label_encoder.transform(
            list(labels)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        token_ids = [
            CLS_ID
        ] + [
            word_to_index.get(
                token,
                UNK_ID,
            )
            for token in tokenize(
                self.texts[index]
            )
        ]

        return {
            "token_ids": torch.tensor(
                token_ids,
                dtype=torch.long,
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
            "text": self.texts[index],
        }


def collate_classification_batch(batch):
    lengths = torch.tensor(
        [
            len(item["token_ids"])
            for item in batch
        ],
        dtype=torch.long,
    )

    maximum_length = int(
        lengths.max().item()
    )

    token_ids = torch.full(
        (
            len(batch),
            maximum_length,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = []

    for row, item in enumerate(batch):
        sequence = item["token_ids"]
        token_ids[
            row,
            :len(sequence),
        ] = sequence
        labels.append(
            item["label"]
        )

    return {
        "token_ids": token_ids,
        "padding_mask": (
            token_ids == PAD_ID
        ),
        "labels": torch.stack(labels),
        "texts": [
            item["text"]
            for item in batch
        ],
    }

In [ ]:
train_dataset = ClassificationDataset(
    X_train,
    y_train,
)
validation_dataset = ClassificationDataset(
    X_validation,
    y_validation,
)
test_dataset = ClassificationDataset(
    X_test,
    y_test,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_classification_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_classification_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_classification_batch,
)

sample_batch = next(iter(train_loader))

print(sample_batch["token_ids"].shape)
print(sample_batch["padding_mask"].shape)

# 23. Positional Encoding Module

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 256,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[:, 0::2] = torch.sin(
            positions * rates
        )
        encoding[:, 1::2] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )

# 24. Transformer Encoder Classifier

In [ ]:
class TransformerTextClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        class_count: int,
        model_dimension: int = 32,
        head_count: int = 4,
        feed_forward_dimension: int = 64,
        layer_count: int = 2,
        dropout: float = 0.15,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = SinusoidalPositionalEncoding(
            model_dimension=model_dimension,
            maximum_length=128,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=feed_forward_dimension,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layer_count,
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            model_dimension,
            class_count,
        )

        self.scale = math.sqrt(
            model_dimension
        )

    def forward(
        self,
        token_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        embedded = (
            self.embedding(token_ids)
            * self.scale
        )

        embedded = self.position(
            embedded
        )

        encoded = self.encoder(
            embedded,
            src_key_padding_mask=padding_mask,
        )

        cls_representation = encoded[
            :,
            0,
            :,
        ]

        logits = self.classifier(
            self.dropout(
                cls_representation
            )
        )

        return {
            "logits": logits,
            "encoded": encoded,
            "representation": cls_representation,
        }

In [ ]:
DEVICE = torch.device("cpu")

torch.manual_seed(42)

model = TransformerTextClassifier(
    vocabulary_size=len(vocabulary),
    class_count=len(label_encoder.classes_),
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
    ),
)

# 25. Shape Inspection

In [ ]:
with torch.no_grad():
    shape_output = model(
        sample_batch[
            "token_ids"
        ].to(DEVICE),
        sample_batch[
            "padding_mask"
        ].to(DEVICE),
    )

print(
    "Encoded:",
    shape_output["encoded"].shape,
)
print(
    "Representation:",
    shape_output["representation"].shape,
)
print(
    "Logits:",
    shape_output["logits"].shape,
)

# 26. Mask Inspection

In [ ]:
mask_frame = pd.DataFrame(
    sample_batch[
        "padding_mask"
    ].int().numpy()
)

mask_frame

`1` indicates a padded position that must be ignored by self-attention.

# 27. Training Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


loss_function = nn.CrossEntropyLoss()


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
):
    model.eval()

    losses = []
    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_texts = []

    with torch.no_grad():
        for batch in loader:
            token_ids = batch[
                "token_ids"
            ].to(DEVICE)
            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)
            labels = batch[
                "labels"
            ].to(DEVICE)

            output = model(
                token_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            probabilities = torch.softmax(
                output["logits"],
                dim=1,
            )

            predictions = probabilities.argmax(
                dim=1
            )

            losses.append(
                float(loss.item())
            )
            all_labels.extend(
                labels.cpu().tolist()
            )
            all_predictions.extend(
                predictions.cpu().tolist()
            )
            all_probabilities.extend(
                probabilities.cpu().tolist()
            )
            all_texts.extend(
                batch["texts"]
            )

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(
            all_labels,
            all_predictions,
        ),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
        ),
        "labels": np.asarray(all_labels),
        "predictions": np.asarray(all_predictions),
        "probabilities": np.asarray(all_probabilities),
        "texts": all_texts,
    }

# 28. Training the Classifier

In [ ]:
def train_classifier(
    model: nn.Module,
    epochs: int = 55,
    learning_rate: float = 0.004,
    weight_decay: float = 1e-4,
    patience: int = 10,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            token_ids = batch[
                "token_ids"
            ].to(DEVICE)
            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)
            labels = batch[
                "labels"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                token_ids,
                padding_mask,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            loss.backward()

            gradient_norm = clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            training_losses.append(
                float(loss.item())
            )
            gradient_norms.append(
                float(gradient_norm)
            )

        validation_metrics = evaluate_model(
            model,
            validation_loader,
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(training_losses)
                ),
                "validation_loss": validation_metrics[
                    "loss"
                ],
                "validation_accuracy": validation_metrics[
                    "accuracy"
                ],
                "validation_macro_f1": validation_metrics[
                    "macro_f1"
                ],
                "mean_gradient_norm": float(
                    np.mean(gradient_norms)
                ),
            }
        )

        if (
            validation_metrics["loss"]
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_metrics["loss"]
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

trained_model, training_history = train_classifier(
    model
)

print(
    "Epochs completed:",
    len(training_history),
)
print(
    "Best validation F1:",
    round(
        training_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 29. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["training_loss"],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Transformer Encoder Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "mean_gradient_norm"
    ],
)
plt.xlabel("Epoch")
plt.ylabel("Mean pre-clipping gradient norm")
plt.title("Transformer Gradient Norms")
plt.tight_layout()
plt.show()

# 30. Evaluation

In [ ]:
test_metrics = evaluate_model(
    trained_model,
    test_loader,
)

print(
    "Test loss:",
    round(test_metrics["loss"], 3),
)
print(
    "Test accuracy:",
    round(test_metrics["accuracy"], 3),
)
print(
    "Test macro F1:",
    round(test_metrics["macro_f1"], 3),
)

In [ ]:
actual_labels = label_encoder.inverse_transform(
    test_metrics["labels"]
)

predicted_labels = label_encoder.inverse_transform(
    test_metrics["predictions"]
)

print(
    classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0,
    )
)

# 31. Confusion Matrix

In [ ]:
class_names = list(
    label_encoder.classes_
)

matrix = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 32. Error Analysis

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": test_metrics["texts"],
        "actual": actual_labels,
        "predicted": predicted_labels,
        "confidence": test_metrics[
            "probabilities"
        ].max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

Review errors for:

- OOV words;
- short ambiguous inputs;
- mixed domains;
- spurious lexical cues;
- insufficient training examples;
- positional truncation.

# 33. Representation Inspection

In [ ]:
trained_model.eval()

with torch.no_grad():
    batch = next(iter(test_loader))

    representation_output = trained_model(
        batch["token_ids"].to(DEVICE),
        batch["padding_mask"].to(DEVICE),
    )

    representations = (
        representation_output[
            "representation"
        ].cpu().numpy()
    )

pd.DataFrame(
    {
        "text": batch["texts"],
        "representation_norm": np.linalg.norm(
            representations,
            axis=1,
        ),
    }
)

# 34. Self-Attention Versus RNNs

In [ ]:
comparison = pd.DataFrame(
    [
        ("Context path", "recurrent chain", "direct token pair"),
        ("Training parallelism", "limited", "high"),
        ("Long-sequence cost", "linear steps", "quadratic attention"),
        ("Position", "implicit in recurrence", "explicit encoding required"),
        ("Streaming", "natural", "requires special design"),
    ],
    columns=["Property", "RNN", "Transformer"],
)

comparison

# 35. Common Failure Modes

- incorrect padding-mask polarity;
- missing positional encoding;
- incompatible model dimension and head count;
- excessive sequence length;
- overfitting small datasets;
- unstable learning rate;
- treating attention weights as guaranteed explanations;
- inadequate tokenization.

In [ ]:
failure_modes = pd.DataFrame(
    [
        ("Mask polarity", "verify blocked positions explicitly"),
        ("No position signal", "add learned or sinusoidal positions"),
        ("OOM on long text", "truncate, chunk, or use efficient attention"),
        ("Overfitting", "dropout, weight decay, early stopping"),
        ("Poor OOV handling", "use subword tokenization"),
    ],
    columns=["Failure", "Response"],
)

failure_modes

# 36. Arabic and Multilingual Considerations

Transformer performance depends heavily on tokenization.

Arabic challenges include:

- clitic attachment;
- rich morphology;
- optional tashkeel;
- orthographic variation;
- MSA and dialects;
- code-switching;
- long subword sequences.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

Subword tokenization can reduce OOV rates but may fragment meaningful morphemes.
For fully vocalized Arabic tasks, tashkeel should be preserved when it is part of
the target representation.

In [ ]:
tokenization_tradeoffs = pd.DataFrame(
    [
        ("Word", "short sequences", "large sparse vocabulary"),
        ("Morphological segment", "linguistic units", "analyzer required"),
        ("Subword", "better coverage", "fragmentation possible"),
        ("Character", "small vocabulary", "very long sequences"),
    ],
    columns=["Unit", "Benefit", "Cost"],
)

tokenization_tradeoffs

# 37. Reproducibility and Reporting

Report:

- dataset and split;
- tokenization;
- vocabulary;
- maximum sequence length;
- model dimension;
- number of heads;
- feed-forward dimension;
- layer count;
- positional encoding;
- normalization strategy;
- dropout;
- optimizer and learning rate;
- gradient clipping;
- early stopping;
- random seed;
- metrics;
- hardware.

In [ ]:
import platform

metadata = pd.Series(
    {
        "dataset_examples": len(dataset),
        "classes": dataset["label"].nunique(),
        "vocabulary_size": len(vocabulary),
        "model_dimension": 32,
        "head_count": 4,
        "encoder_layers": 2,
        "feed_forward_dimension": 64,
        "position_encoding": "sinusoidal",
        "normalization": "pre-norm",
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "torch_version": torch.__version__,
    },
    name="Transformer encoder experiment",
)

metadata

# 38. Knowledge Check

1. What is self-attention?
2. How are queries, keys, and values produced?
3. Why is dot-product attention scaled?
4. What does a padding mask block?
5. What does a causal mask block?
6. How does multi-head attention differ from single-head attention?
7. Why is positional information required?
8. How do sinusoidal and learned positions differ?
9. What is the purpose of residual connections?
10. What does layer normalization normalize?
11. What does the feed-forward network do?
12. How does a Transformer encoder block operate?
13. How do pre-norm and post-norm differ?
14. Why is full self-attention quadratic?
15. How does Arabic tokenization affect Transformer inputs?

# 39. Exercises

## Exercise 1 — Manual Attention

Calculate one scaled dot-product attention example by hand.

## Exercise 2 — Masking

Combine padding and causal masks.

## Exercise 3 — Multi-Head Shapes

Trace all tensor shapes for eight heads.

## Exercise 4 — Positional Encoding

Compare learned and sinusoidal positions.

## Exercise 5 — Head Count

Train models with one, two, and four heads.

## Exercise 6 — Layer Count

Compare one-layer and two-layer encoders.

## Exercise 7 — Pooling

Compare CLS, mean pooling, and max pooling.

## Exercise 8 — Long Text

Test truncation and chunking strategies.

## Exercise 9 — Arabic Classification

Compare word, subword, and character representations.

## Exercise 10 — Efficient Attention

Research sparse and linear attention approaches.

## Challenge Exercises

1. Implement multi-head attention directly with PyTorch tensor operations.
2. Return and visualize attention weights from a custom encoder block.
3. Implement a causal Transformer language model.
4. Add label smoothing and a learning-rate warmup schedule.
5. Compare Transformer, GRU, and LSTM classifiers under matched parameter budgets.

# 40. Summary and Next Lesson

In this lesson:

- self-attention replaced recurrence with direct token-to-token interaction;
- query, key, and value projections were implemented;
- scaled dot-product attention was calculated;
- padding and causal masks were distinguished;
- multi-head attention and head concatenation were explained;
- sinusoidal positional encoding represented token order;
- residual connections and layer normalization supported optimization;
- position-wise feed-forward layers transformed token features;
- Transformer encoder blocks were assembled conceptually;
- a complete CPU-only Transformer text classifier was trained;
- learning curves, metrics, errors, and representations were inspected;
- Transformer and recurrent computational properties were compared;
- Arabic morphology, segmentation, subwords, and tashkeel were connected to
  Transformer input design.

## Next Lesson

**Lesson 32: Transformer Encoder Models and Contextual Embeddings** introduces
contextual token representations, masked language modeling, encoder-only
architectures, subword inputs, sentence pooling, and downstream fine-tuning.

# References

- Vaswani, A. et al. *Attention Is All You Need*.
- Ba, J. L., Kiros, J. R., & Hinton, G. E. layer normalization.
- Devlin, J. et al. BERT literature.
- Goodfellow, I., Bengio, Y., & Courville, A. *Deep Learning*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.